# Июль 2026: полнота данных + 10 эталонных ИНН

Отдельная справка для коллег (не часть основной сборки витрины).

## Зачем
1. **Июль в витрине предварительный**: Excel-отчёта нет; многие источники догружаются в следующем месяце.
2. Показать **насколько июль «недогружен»** относительно июня / среднего Jan–Jun.
3. Дать **10 ИНН**, где Jan–Jun lake = Excel и есть ненулевая активность каждый месяц (эталон для доверия к витрине).

## Входы
- `final_df_period_2026_01_2026_07_mpos.csv` (после пересборки)
- Excel `01_Январь` … `06_Июнь_2026.xlsx`

## Выходы
- `july_2026_data_completeness_brief.md`
- `july_vs_june_coverage_metrics.csv`
- `stable_excel_match_inns_jan_jun_examples.csv`


In [ ]:
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')
OUTPUT_DIR = DATA_DIR

JULY = '2026-07'
JUNE = '2026-06'
MATCH_MONTHS = ['2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06']
MONEY_TOL = 0.01
N_EXAMPLES = 10

excel_reference_by_month = {
    '2026-01': DATA_DIR / '01_Январь_2026.xlsx',
    '2026-02': DATA_DIR / '02_Февраль_2026.xlsx',
    '2026-03': DATA_DIR / '03_Март_2026.xlsx',
    '2026-04': DATA_DIR / '04_Апрель_2026.xlsx',
    '2026-05': DATA_DIR / '05_Май_2026.xlsx',
    '2026-06': DATA_DIR / '06_Июнь_2026.xlsx',
}
excel_header_by_month = {
    '2026-01': 1,
    '2026-02': 1,
    '2026-03': 0,
    '2026-04': 0,
    '2026-05': 0,
    '2026-06': 0,
}

FINAL_DF_CANDIDATES = [
    DATA_DIR / 'final_df_period_2026_01_2026_07_mpos.csv',
    DATA_DIR / 'final_df_period_2026_01_2026_06_mpos.csv',
]

COVERAGE_METRICS = [
    'unique_inn', 'retl_cnt', 'term_cnt', 'trx_cnt', 'trx_sum',
    'commission_from_ops', 'commission_monthly', 'int_component',
    'chod', 'aur', 'amortization', 'fin_result',
]

COUNT_EXACT = ['retl_cnt', 'term_cnt', 'trx_cnt']
MONEY_EXACT = ['trx_sum', 'commission_monthly', 'chod']


def normalize_inn_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def normalize_agr_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None'}:
        return None
    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass
    s = re.sub(r'\.0$', '', s)
    return s if s not in {'', 'nan', 'None'} else None


def pick_col_robust(columns, candidates):
    cols = list(columns)
    lower = {str(c).strip().lower(): c for c in cols}
    for cand in candidates:
        key = str(cand).strip().lower()
        if key in lower:
            return lower[key]
    for c in cols:
        cl = str(c).strip().lower().replace('\n', ' ')
        for cand in candidates:
            if str(cand).strip().lower() in cl:
                return c
    return None


def to_num_series(s):
    return pd.to_numeric(
        s.astype(str)
         .str.replace('\xa0', '', regex=False)
         .str.replace(' ', '', regex=False)
         .str.replace(',', '.', regex=False),
        errors='coerce',
    )


print('DATA_DIR:', DATA_DIR)
print('Excel:')
for m, p in excel_reference_by_month.items():
    print(f'  {m}: exists={p.exists()} header={excel_header_by_month.get(m, 0)} | {p.name}')
print('final_df candidates:')
for p in FINAL_DF_CANDIDATES:
    print(f'  exists={p.exists()} | {p}')


## 1) Загрузка `final_df` + Excel Jan–Jun


In [ ]:
# --- load final_df ---
fdf_path = None
for pth in FINAL_DF_CANDIDATES:
    if pth.exists():
        fdf_path = pth
        break
if fdf_path is None:
    raise FileNotFoundError(
        'final_df CSV не найден. Сначала дождитесь пересборки '
        '01_07_acq_dash_jan_jun_mpos.ipynb → final_df_period_2026_01_2026_07_mpos.csv'
    )

lake_df = pd.read_csv(fdf_path, dtype=str, low_memory=False)
lake_df['report_month'] = lake_df['report_month'].astype(str).str.slice(0, 7)
if 'inn_key' not in lake_df.columns:
    inn_src = 'inn' if 'inn' in lake_df.columns else None
    if inn_src is None:
        raise RuntimeError('final_df: нет inn / inn_key')
    lake_df['inn_key'] = lake_df[inn_src].map(normalize_inn_q1)
else:
    lake_df['inn_key'] = lake_df['inn_key'].map(normalize_inn_q1)

for c in COVERAGE_METRICS + COUNT_EXACT + MONEY_EXACT + ['active_retl_cnt', 'active_terms', 'term_cnt']:
    if c in lake_df.columns:
        lake_df[c] = pd.to_numeric(lake_df[c], errors='coerce')

print(f'Loaded lake: {fdf_path.name} | rows={len(lake_df):,} | months={sorted(lake_df["report_month"].unique())}')


COL_MAP = {
    'inn_col': ['ИНН', 'inn', 'c_inn'],
    'agr_col': ['ID договора', 'Номер договора', 'agr_id', 'abs_agr_id'],
    'retl_col': ['Кол-во торговых точек', 'Ко-во торговых точек', 'Количество торговых точек'],
    'term_col': ['Кол-во терминалов', 'Количество терминалов'],
    'trx_cnt_col': ['Количество операций', 'Количеств операций', 'trx_cnt'],
    'trx_sum_col': ['Сумма операций', 'Сумма опреаций', 'trx_sum'],
    'comm_ops_col': [
        'Комиссия эквайринга', 'Комиссия (% с операций)',
        'Комиссия \n(% с операций)', 'Комиссия % с операций',
    ],
    'comm_monthly_col': [
        'Комиссия в месяц', 'Комиссия CN (₽ в месяц)', 'Комиссия (₽ в месяц)',
        'Комиссия \n(₽ в месяц)', 'Комиссия (руб в месяц)',
    ],
    'int_component_col': [
        'Комиссия МПС (IRF, ₽)', 'Комиссия МПС (IRF, р)',
        'Комиссия МПС (IRF, руб)', 'Комиссия МПС (IRF)',
    ],
    'chod_col': ['ЧОД'],
    'aur_col': ['АУР', 'AUR', 'Aur', 'Аур'],
    'amortization_col': [
        'Амортизация', 'Аморт', 'Амортизация терминалов',
        'amortization', 'Amortization', 'Амортизация, руб',
    ],
    'fin_result_col': [
        'Фин. Рез.', 'Фин.Рез.', 'Фин.рез.', 'Фин. рез.',
        'Фин рез', 'Финрез', 'Фин результат', 'Финансовый результат',
        'fin_result', 'Fin.Res.', 'FinRes',
    ],
}


def load_excel_month(report_month, excel_path, excel_header=0):
    ex = pd.read_excel(excel_path, header=excel_header)
    resolved = {k: pick_col_robust(ex.columns, v) for k, v in COL_MAP.items()}
    required = ['inn_col', 'agr_col', 'chod_col']
    missing = [k for k in required if resolved.get(k) is None]
    if missing:
        raise ValueError(f'{report_month}: missing {missing}. columns={list(ex.columns)}')

    out = pd.DataFrame({
        'report_month': report_month,
        'inn_key': ex[resolved['inn_col']].map(normalize_inn_q1),
        'agr_id_key': ex[resolved['agr_col']].map(normalize_agr_q1),
    })

    def take(name, dest, default=np.nan):
        col = resolved.get(name)
        if col is None:
            out[dest] = default
        else:
            out[dest] = to_num_series(ex[col])

    take('retl_col', 'retl_cnt')
    take('term_col', 'term_cnt')
    take('trx_cnt_col', 'trx_cnt')
    take('trx_sum_col', 'trx_sum')
    take('comm_ops_col', 'commission_from_ops')
    take('comm_monthly_col', 'commission_monthly')
    take('int_component_col', 'int_component')
    take('chod_col', 'chod')
    take('aur_col', 'aur')
    take('amortization_col', 'amortization')
    take('fin_result_col', 'fin_result')

    agr = (
        out.dropna(subset=['agr_id_key'])
        .groupby(['report_month', 'inn_key', 'agr_id_key'], as_index=False)
        .agg({
            'retl_cnt': 'max',
            'term_cnt': 'max',
            'trx_cnt': 'max',
            'trx_sum': 'sum',
            'commission_from_ops': 'sum',
            'commission_monthly': 'sum',
            'int_component': 'sum',
            'chod': 'sum',
            'aur': 'sum',
            'amortization': 'sum',
            'fin_result': 'sum',
        })
    )
    return agr, resolved


excel_parts = []
for month, path in excel_reference_by_month.items():
    if not path.exists():
        print('SKIP missing Excel:', month, path)
        continue
    header = int(excel_header_by_month.get(month, 0))
    agr_df, resolved = load_excel_month(month, path, excel_header=header)
    excel_parts.append(agr_df)
    print(
        f'{month}: agr_rows={len(agr_df):,} | chod={agr_df["chod"].fillna(0).sum():,.0f} '
        f'| header={header}'
    )

if not excel_parts:
    raise RuntimeError('No Excel months loaded')

excel_agr_df = pd.concat(excel_parts, ignore_index=True)
print('excel_agr_df rows =', len(excel_agr_df))
display(lake_df.head(2))
display(excel_agr_df.head(2))


## 2) Справка: насколько полон июль

Июль — **lake-only**. Сравниваем тоталы июля с июнем и со средним Jan–Jun.
Ориентир полноты: `% от июня` по `trx_cnt` / `trx_sum` / `chod`.


In [ ]:
def lake_month_totals(df):
    rows = []
    for month, g in df.groupby('report_month'):
        row = {'report_month': month}
        row['unique_inn'] = g['inn_key'].nunique()
        row['agr_rows'] = g.shape[0]
        for m in COVERAGE_METRICS:
            if m == 'unique_inn':
                continue
            if m in g.columns:
                row[m] = float(pd.to_numeric(g[m], errors='coerce').fillna(0).sum())
            else:
                row[m] = np.nan
        rows.append(row)
    return pd.DataFrame(rows).sort_values('report_month').reset_index(drop=True)


lake_monthly = lake_month_totals(lake_df)
print('=== Lake monthly totals ===')
display(lake_monthly)

if JULY not in set(lake_monthly['report_month']):
    raise RuntimeError(
        f'В final_df нет месяца {JULY}. Есть: {sorted(lake_monthly["report_month"].unique())}. '
        'Дождитесь пересборки с July или укажите актуальный CSV.'
    )

july_row = lake_monthly.loc[lake_monthly['report_month'] == JULY].iloc[0]
has_june = JUNE in set(lake_monthly['report_month'])
june_row = lake_monthly.loc[lake_monthly['report_month'] == JUNE].iloc[0] if has_june else None

hist = lake_monthly.loc[lake_monthly['report_month'].isin(MATCH_MONTHS)].copy()
hist_avg = hist[COVERAGE_METRICS].mean(numeric_only=True)

cmp_rows = []
for m in COVERAGE_METRICS:
    july_v = float(july_row.get(m, np.nan)) if pd.notna(july_row.get(m, np.nan)) else np.nan
    june_v = float(june_row.get(m, np.nan)) if (june_row is not None and pd.notna(june_row.get(m, np.nan))) else np.nan
    avg_v = float(hist_avg.get(m, np.nan)) if pd.notna(hist_avg.get(m, np.nan)) else np.nan
    pct_june = (july_v / june_v * 100.0) if (pd.notna(july_v) and pd.notna(june_v) and june_v != 0) else np.nan
    pct_avg = (july_v / avg_v * 100.0) if (pd.notna(july_v) and pd.notna(avg_v) and avg_v != 0) else np.nan
    cmp_rows.append({
        'metric': m,
        'july': july_v,
        'june': june_v,
        'avg_jan_jun': avg_v,
        'july_pct_of_june': pct_june,
        'july_pct_of_avg_jan_jun': pct_avg,
        'delta_july_minus_june': july_v - june_v if (pd.notna(july_v) and pd.notna(june_v)) else np.nan,
    })

coverage_cmp = pd.DataFrame(cmp_rows)
print('=== July vs June / avg Jan–Jun ===')
display(coverage_cmp)

# key completeness ratios
def _pct(metric):
    r = coverage_cmp.loc[coverage_cmp['metric'] == metric, 'july_pct_of_june']
    return float(r.iloc[0]) if len(r) and pd.notna(r.iloc[0]) else np.nan

pct_trx_cnt = _pct('trx_cnt')
pct_trx_sum = _pct('trx_sum')
pct_chod = _pct('chod')
pct_comm_m = _pct('commission_monthly')
pct_inn = _pct('unique_inn')

anchor = np.nanmean([x for x in [pct_trx_sum, pct_chod, pct_trx_cnt] if pd.notna(x)])
if pd.isna(anchor):
    verdict_line = (
        'Июль 2026 в витрине — предварительный (lake-only, без Excel). '
        'Не удалось посчитать % от июня — проверьте наличие июня в final_df.'
    )
else:
    verdict_line = (
        f'Июль 2026 в витрине — **не финальный** (нет Excel; источники догружаются следующим месяцем). '
        f'Ориентир полноты относительно июня: оборот `trx_sum` ≈ **{pct_trx_sum:.0f}%**, '
        f'ЧОД ≈ **{pct_chod:.0f}%**, операции ≈ **{pct_trx_cnt:.0f}%**, '
        f'ИНН ≈ **{pct_inn:.0f}%**, `commission_monthly` ≈ **{pct_comm_m:.0f}%**. '
        f'Не использовать июль для жёсткой сверки с бизнесом до догрузки.'
    )

print('=== VERDICT ===')
print(verdict_line)
display(Markdown('### Вердикт для коллег\n\n' + verdict_line))

coverage_cmp.to_csv(OUTPUT_DIR / 'july_vs_june_coverage_metrics.csv', index=False, encoding='utf-8-sig')
print('Saved', OUTPUT_DIR / 'july_vs_june_coverage_metrics.csv')


## 3) 10 эталонных ИНН (Jan–Jun = Excel + динамика)

Критерии:
1. ИНН есть во **всех** месяцах Jan–Jun в lake и в Excel.
2. На зерне ИНН совпадают: `retl_cnt`, `term_cnt`, `trx_cnt` (точно); `trx_sum`, `commission_monthly`, `chod` (|Δ|≤0.01).
3. `trx_cnt > 0` и `trx_sum > 0` в **каждом** месяце Jan–Jun.
4. Берём топ-10 по среднему `trx_sum` Jan–Jun.


In [ ]:
def agg_inn_month(df, value_cols, count_how='sum'):
    """Aggregate agr-level → INN × month."""
    use = df.dropna(subset=['inn_key', 'report_month']).copy()
    ag = {'agr_n': ('inn_key', 'size')}
    for c in value_cols:
        if c not in use.columns:
            continue
        if c in ('retl_cnt', 'term_cnt'):
            ag[c] = (c, 'sum')
        else:
            ag[c] = (c, 'sum')
    # named agg via groupby.agg dict style
    g = use.groupby(['report_month', 'inn_key'], as_index=False)
    out = g.agg(
        agr_n=('inn_key', 'size'),
        **{
            c: (c, 'sum')
            for c in value_cols
            if c in use.columns
        },
    )
    return out


LAKE_INN_COLS = ['retl_cnt', 'term_cnt', 'trx_cnt', 'trx_sum', 'commission_monthly', 'chod', 'fin_result']
EX_INN_COLS = ['retl_cnt', 'term_cnt', 'trx_cnt', 'trx_sum', 'commission_monthly', 'chod', 'fin_result']

lake_jj = lake_df.loc[lake_df['report_month'].isin(MATCH_MONTHS)].copy()
excel_jj = excel_agr_df.loc[excel_agr_df['report_month'].isin(MATCH_MONTHS)].copy()

lake_inn = agg_inn_month(lake_jj, LAKE_INN_COLS)
excel_inn = agg_inn_month(excel_jj, EX_INN_COLS)

# presence all 6 months
lake_months_n = lake_inn.groupby('inn_key')['report_month'].nunique()
excel_months_n = excel_inn.groupby('inn_key')['report_month'].nunique()
inns_all_lake = set(lake_months_n[lake_months_n >= len(MATCH_MONTHS)].index)
inns_all_excel = set(excel_months_n[excel_months_n >= len(MATCH_MONTHS)].index)
inns_both = inns_all_lake & inns_all_excel
print(f'INNs in all 6 months: lake={len(inns_all_lake):,} excel={len(inns_all_excel):,} both={len(inns_both):,}')

lk = lake_inn.loc[lake_inn['inn_key'].isin(inns_both)].copy()
ex = excel_inn.loc[excel_inn['inn_key'].isin(inns_both)].copy()

merged = lk.merge(
    ex,
    on=['report_month', 'inn_key'],
    how='inner',
    suffixes=('_lake', '_excel'),
)
print('merged inn×month rows:', len(merged))


def month_matches(row):
    for c in COUNT_EXACT:
        a = row.get(f'{c}_lake')
        b = row.get(f'{c}_excel')
        if pd.isna(a) and pd.isna(b):
            continue
        if float(a or 0) != float(b or 0):
            return False
    for c in MONEY_EXACT:
        a = float(row.get(f'{c}_lake') or 0)
        b = float(row.get(f'{c}_excel') or 0)
        if abs(a - b) > MONEY_TOL:
            return False
    return True


merged['month_ok'] = merged.apply(month_matches, axis=1)
merged['has_dyn'] = (
    pd.to_numeric(merged['trx_cnt_lake'], errors='coerce').fillna(0) > 0
) & (
    pd.to_numeric(merged['trx_sum_lake'], errors='coerce').fillna(0) > 0
)

per_inn = (
    merged.groupby('inn_key', as_index=False)
    .agg(
        months_n=('report_month', 'nunique'),
        months_ok=('month_ok', 'sum'),
        months_dyn=('has_dyn', 'sum'),
        avg_trx_sum=('trx_sum_lake', 'mean'),
        sum_trx_sum=('trx_sum_lake', 'sum'),
        avg_chod=('chod_lake', 'mean'),
    )
)

candidates = per_inn.loc[
    (per_inn['months_n'] >= len(MATCH_MONTHS))
    & (per_inn['months_ok'] >= len(MATCH_MONTHS))
    & (per_inn['months_dyn'] >= len(MATCH_MONTHS))
].copy()

candidates = candidates.sort_values('avg_trx_sum', ascending=False)
print(f'Candidates (exact match + dynamics all months): {len(candidates):,}')
display(candidates.head(15))

if len(candidates) < N_EXAMPLES:
    print(
        f'WARNING: только {len(candidates)} кандидатов < {N_EXAMPLES}. '
        'Возьмём всех; при необходимости ослабьте MONEY_EXACT / уберите commission_monthly.'
    )

top_inns = candidates.head(N_EXAMPLES)['inn_key'].tolist()
print('Selected INNs:', top_inns)

# detail table for examples
detail_parts = []
for rank, inn in enumerate(top_inns, start=1):
    sub = merged.loc[merged['inn_key'] == inn].sort_values('report_month').copy()
    sub.insert(0, 'rank', rank)
    detail_parts.append(sub)

if detail_parts:
    examples_wide = pd.concat(detail_parts, ignore_index=True)
else:
    examples_wide = pd.DataFrame()

# compact view for colleagues
compact_cols = [
    'rank', 'inn_key', 'report_month',
    'trx_cnt_lake', 'trx_sum_lake', 'retl_cnt_lake', 'term_cnt_lake', 'chod_lake',
    'trx_cnt_excel', 'trx_sum_excel', 'retl_cnt_excel', 'term_cnt_excel', 'chod_excel',
    'month_ok',
]
compact_cols = [c for c in compact_cols if c in examples_wide.columns]
examples_compact = examples_wide[compact_cols].copy() if len(examples_wide) else examples_wide

print('=== 10 example INNs (monthly) ===')
display(examples_compact)

# one row per INN summary
summary = candidates.head(N_EXAMPLES)[
    ['inn_key', 'avg_trx_sum', 'sum_trx_sum', 'avg_chod', 'months_ok', 'months_dyn']
].copy()
summary.insert(0, 'rank', range(1, len(summary) + 1))
print('=== Summary ===')
display(summary)

out_csv = OUTPUT_DIR / 'stable_excel_match_inns_jan_jun_examples.csv'
examples_compact.to_csv(out_csv, index=False, encoding='utf-8-sig')
summary.to_csv(OUTPUT_DIR / 'stable_excel_match_inns_jan_jun_summary.csv', index=False, encoding='utf-8-sig')
print('Saved', out_csv)
print('Saved', OUTPUT_DIR / 'stable_excel_match_inns_jan_jun_summary.csv')

# keep for brief
stable_inns_summary = summary
stable_inns_detail = examples_compact
n_candidates = len(candidates)


## 4) Сохранение markdown-справки для коллег


In [ ]:
from io import StringIO

brief_path = OUTPUT_DIR / 'july_2026_data_completeness_brief.md'
buf = StringIO()

def w(line=''):
    buf.write(line + '\n')

w('# Справка: полнота данных за июль 2026 + эталонные ИНН')
w()
w('> Для коллег. Источник: витрина acquiring dashboard (`final_df`), сверка с Excel Jan–Jun.')
w()
w('## 1. Июль — предварительные данные')
w()
w('- В витрине за **июль 2026** есть только расчёт из **озера** (Excel-отчёта за июль нет).')
w('- Часть источников (аренда/MPOS `commission_monthly`, догрузки trx/IRF, закрытия точек/терминалов) обновляется **в следующем месяце**.')
w('- Поэтому KPI июля **нельзя** считать финальными и жёстко сверять с бизнесом до догрузки.')
w()
w('## 2. Ориентир полноты (июль vs июнь)')
w()
w(f'- Файл витрины: `{fdf_path.name}`')
w()
w('| Метрика | Июль | Июнь | Июль / июнь, % |')
w('|---|---:|---:|---:|')
for m in ['unique_inn', 'trx_cnt', 'trx_sum', 'chod', 'commission_monthly', 'retl_cnt', 'term_cnt', 'aur', 'amortization']:
    r = coverage_cmp.loc[coverage_cmp['metric'] == m]
    if r.empty:
        continue
    rr = r.iloc[0]
    def fmt(x):
        return '—' if pd.isna(x) else f'{x:,.0f}'
    def fmtp(x):
        return '—' if pd.isna(x) else f'{x:.0f}%'
    w(f"| `{m}` | {fmt(rr['july'])} | {fmt(rr['june'])} | {fmtp(rr['july_pct_of_june'])} |")
w()
w(f'**Вердикт:** {verdict_line}')
w()
w('## 3. Десять эталонных ИНН (Jan–Jun)')
w()
w('Критерии отбора:')
w('1. ИНН присутствует во всех месяцах Jan–Jun в lake и в Excel;')
w('2. на зерне ИНН совпадают `retl_cnt` / `term_cnt` / `trx_cnt` и `trx_sum` / `commission_monthly` / `chod` (|Δ|≤0.01);')
w(f'3. ненулевые `trx_cnt` и `trx_sum` в каждом месяце Jan–Jun;')
w('4. топ по среднему обороту `trx_sum`.')
w()
w(f'Кандидатов всего: **{n_candidates}**. Ниже — выбранные **{len(stable_inns_summary)}**.')
w()
if len(stable_inns_summary):
    w('| # | ИНН | avg trx_sum | sum trx_sum | avg chod |')
    w('|---:|---|---:|---:|---:|')
    for _, r in stable_inns_summary.iterrows():
        w(
            f"| {int(r['rank'])} | `{r['inn_key']}` | {r['avg_trx_sum']:,.0f} | "
            f"{r['sum_trx_sum']:,.0f} | {r['avg_chod']:,.0f} |"
        )
    w()
    w('Помесячная детализация (lake; при отборе = Excel):')
    w()
    show_cols = ['rank', 'inn_key', 'report_month', 'trx_cnt_lake', 'trx_sum_lake', 'retl_cnt_lake', 'term_cnt_lake', 'chod_lake']
    show_cols = [c for c in show_cols if c in stable_inns_detail.columns]
    mini = stable_inns_detail[show_cols].copy()
    w('| ' + ' | '.join(show_cols) + ' |')
    w('|' + '|'.join(['---'] * len(show_cols)) + '|')
    for _, r in mini.iterrows():
        cells_row = []
        for c in show_cols:
            v = r[c]
            if c in {'trx_sum_lake', 'chod_lake'} and pd.notna(v):
                cells_row.append(f'{float(v):,.0f}')
            elif c in {'trx_cnt_lake', 'retl_cnt_lake', 'term_cnt_lake', 'rank'} and pd.notna(v):
                cells_row.append(f'{float(v):,.0f}')
            else:
                cells_row.append(str(v))
        w('| ' + ' | '.join(cells_row) + ' |')
else:
    w('_Не найдено ИНН, удовлетворяющих всем критериям. Проверьте допуски / наличие Excel и final_df._')
w()
w('## Файлы')
w()
w(f'- `{OUTPUT_DIR / "july_vs_june_coverage_metrics.csv"}`')
w(f'- `{OUTPUT_DIR / "stable_excel_match_inns_jan_jun_examples.csv"}`')
w(f'- `{OUTPUT_DIR / "stable_excel_match_inns_jan_jun_summary.csv"}`')
w(f'- эта справка: `{brief_path}`')

text = buf.getvalue()
brief_path.write_text(text, encoding='utf-8')
print('Saved', brief_path)
display(Markdown(text))
